## Initialization
Make sure to run this under the venv


In [ ]:
import sys
from pathlib import Path

notebook_path = Path().resolve()
project_root = notebook_path.parent
sys.path.append(str(project_root))

## Load files from test-repo
This requires a test-repo (React Repo) to be present in tests/

In [ ]:
from src.schemas import FileObject
from tests.utils import read_repo_files

repo_path: Path = notebook_path / "test-repo"
project_path = project_root / "tests" / "test-repo"
if not repo_path.exists():
    raise FileNotFoundError(f"Test repo not found at {repo_path}")

files: list[FileObject] = await read_repo_files(str(repo_path))

print(f"Read {len(files)} files")

## Main logic
Generating AST for a file

In [ ]:
from esprima import parseModule


def generate_ast(file_path):
    with open(file_path, "r") as file:
        content = file.read()

    try:
        ast = parseModule(content, jsx=True)
        return ast.toDict()
    except Exception as e:
        print(f"Error parsing {file_path}: {str(e)}")
        return None

### All files

In [ ]:
import os


def traverse_directory(directory):
    ignored_dirs = set()
    with open(project_path / ".gitignore", "r") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                ignored_dirs.add(line.strip("/"))

    ignored_files = set()
    with open(project_path / ".gitignore", "r") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                ignored_files.add(line)

    asts = {}
    for root, dirs, files in os.walk(directory):
        for ignored_dir in ignored_dirs:
            if ignored_dir in dirs:
                dirs.remove(ignored_dir)

        for file in files:
            if (
                file.endswith((".js", ".jsx", ".ts", ".tsx"))
                and file not in ignored_files
            ):
                file_path = os.path.join(root, file)
                ast = generate_ast(file_path)
                if ast:
                    asts[file_path] = ast
    return asts


asts = traverse_directory(project_path)

##### Output

In [ ]:
import json

output_file = "outputs/react_project_asts.json"
with open(output_file, "w") as f:
    json.dump(asts, f, indent=2)

print(f"ASTs have been generated and saved to {output_file}")

### Individual file

In [ ]:
file = f"{project_path}/app/page.js"
ast = generate_ast(file)
print(ast)

##### Output

In [ ]:
import json

output_file = f"outputs/ast_{file.split('/')[-1]}.json"
with open(output_file, "w") as f:
    json.dump(ast, f, indent=2)

print(f"AST has been generated and saved to {output_file}")